# ACE-Net Shard Preprocessor
### Assigned: **EL (Account 4)** | `geueljohn.rivera.lexmeet@gmail.com`
### Target: **MELD** | **`shard_0005`** (Round 1)
### Output Target: `Google Drive > THESIS_MOTHERFILE > Baseline preprocessed > MELD`

## Step 1: Connect to GPU & Mount Google Drive

In [ ]:
from google.colab import drive
import os, sys, torch

drive.mount('/content/drive')
print('GPU Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

## Step 2: Clone Repository & Checkout Branch

In [ ]:
%cd /content
!rm -rf Baseline_Training
!git clone https://github.com/gjvlio/Baseline_Training.git
%cd Baseline_Training
!git checkout feat/training-and-preprocessing
!git pull
!git log --oneline -1

## Step 3: Install Required Dependencies

In [ ]:
!pip -q install openai-whisper transformers facenet-pytorch librosa opencv-python tqdm pandas
print('Dependencies installed successfully!')

## Step 4: Unzip Raw Dataset (`meld_raw.zip`) to Local Colab SSD

In [ ]:
import os, zipfile, shutil

DRIVE_ZIP = '/content/drive/MyDrive/THESIS_MOTHERFILE/datasets/meld_raw.zip'
LOCAL_RAW = '/content/data/raw/MELD'

os.makedirs(LOCAL_RAW, exist_ok=True)
if not os.path.exists(DRIVE_ZIP):
    raise FileNotFoundError(f'Raw zip not found in Drive: {DRIVE_ZIP}')

print(f'Unzipping {DRIVE_ZIP} to local SSD ({LOCAL_RAW})...')
with zipfile.ZipFile(DRIVE_ZIP, 'r') as z:
    z.extractall(LOCAL_RAW)
print('Unzip complete! Local files ready.')

## Step 5: Execute Preprocessing for `MELD` [shard_0005]

In [ ]:
!python scripts/preprocess/run_shard.py \
    --account 'geueljohn.rivera.lexmeet@gmail.com' \
    --dataset 'MELD' \
    --shard '0005' \
    --raw_dir '/content/data/raw/MELD' \
    --drive_root '/content/drive/MyDrive/THESIS_MOTHERFILE' \
    --device cuda

## Step 6: Post-Run Integrity Check & Verification

In [ ]:
import json, glob
from pathlib import Path

ckpt_file = Path('/content/drive/MyDrive/THESIS_MOTHERFILE/Baseline preprocessed/MELD/checkpoints/shard_0005_checkpoint.json')
out_shard = Path('/content/drive/MyDrive/THESIS_MOTHERFILE/Baseline preprocessed/MELD/shards/shard_0005')

if ckpt_file.exists():
    with open(ckpt_file) as f:
        data = json.load(f)
    print('=' * 60)
    print('SHARD STATUS:', data.get('status'))
    print('Completed Clips:', len(data.get('completed_ids', [])))
    print('Failed Clips   :', len(data.get('failed_ids', [])))
    print('=' * 60)
else:
    print('Checkpoint not found. Run Step 5 first.')